# Encoder probe - is representation quality the bottleneck? (no training)

**Why this exists.** The xattn model overfits hard: on split-42, `A160r_s42` drives
train loss 0.61 -> 0.10 while val loss goes 0.54 -> 1.52, and the best epoch is 6 of 11.
So *training* a 5x bigger encoder end-to-end is the wrong experiment - it would memorise faster.
The open question is narrower: **is the frozen representation the bottleneck?**

This notebook answers that with zero training and zero overfitting risk:

1. load each encoder (fp16, no gradient), mean-pool the last hidden state, L2-normalise;
2. fit one logistic regression (fixed C=0.1, no val tuning) on the 6,889 train rows;
3. report split-42 val macro F1 / per-language F1.

Candidates:

| tag | checkpoint | params | dim | why |
|---|---|---|---|---|
| `minilm` | `paraphrase-multilingual-MiniLM-L12-v2` | 118M | 384 | the current backbone (control) |
| `mmbert_base` | `jhu-clsp/mmbert-base` | 307M | 768 | the model the user suspects overfits |
| `e5_large` | `intfloat/multilingual-e5-large` | 560M | 1024 | 4.7x capacity, multilingual |
| `bge_m3` | `BAAI/bge-m3` | 568M | 1024 | 4.8x capacity, 100+ languages |

Text variants: `comment` / `title+comment` / `title+desc+comment`. If context matters, the
last one wins; if the encoder is the bottleneck, the 1024-dim models win on all three.

**How to read it.** The control (`minilm`, `title+desc+comment`) is the number to beat. A
+1.5pt or better probe at the same text variant is real evidence that the frozen representation
is limiting, and the next xattn run should use that encoder **with the backbone frozen**
(`--unfreeze-layers 0`) so capacity cannot turn into memorisation. A flat probe closes the
encoder track for good.

Caveat: mean pooling + linear probe is not the xattn architecture (no cross-attention readout),
so absolute numbers sit below .7476. Only the *relative* comparison is meaningful.


In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/AmnO-O/Try.git'
PIN_COMMIT = '37c4c2645aea0dff1e749dc6f762f33d5e843d35'


def sh(cmd):
    print(f'$ {cmd}', flush=True)
    subprocess.run(cmd, shell=True, check=True)


if os.path.basename(os.path.abspath('.')) != 'Try':
    if not os.path.isdir('Try'):
        sh(f'git clone {REPO_URL}')
    os.chdir('Try')
if PIN_COMMIT:
    sh('git fetch origin main')
    sh(f'git checkout {PIN_COMMIT}')
sh('git log -1 --format="%H %cd %s" --date=iso')
sh('pip install -q -r requirements.txt')
print('cwd:', os.getcwd(), flush=True)


## 1. Data (same split-42 frame as every other batch)

In [ ]:
import glob
import os
import sys

hits = sorted(glob.glob('/kaggle/input/**/StereoQueerEval_*_training.tsv', recursive=True))
assert hits, 'StereoQueerEval TSVs not found - attach the dataset (Add Input > amnpham/lgbtqia)'
os.environ['LGBT_DATA_DIR'] = os.path.dirname(hits[0])
os.environ['LGBT_WEIGHTS_DIR'] = os.path.abspath('weights')
os.makedirs('weights', exist_ok=True)

import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), 'No GPU - set Settings > Accelerator > GPU T4 x2'
sys.path.insert(0, os.path.abspath('src'))
import data as data_mod

df_all = data_mod.load_frame(languages=['EN', 'IT', 'NL'])
df_train, df_val = data_mod.split_by_video(df_all, 0.1, 42)
labs = df_val['st_y'].astype(int).to_numpy()
assert len(labs) == 738 and int(labs.sum()) == 276
np.save('weights/val_labels.npy', labs)
np.save('weights/val_langs.npy', df_val['lang'].to_numpy(dtype=str))
print('train', len(df_train), '| val', len(df_val), '| pos', int(labs.sum()))


## 2. Frozen embeddings

One model in GPU memory at a time. Each (encoder, variant) writes two `.npy` files and is
skipped on re-run, so a crash does not lose earlier work.


In [ ]:
import gc
import time

import torch.nn.functional as F
from model_xattn import encoder_model_class
from transformers import AutoTokenizer

DEV = 'cuda:0'
MAXLEN = 256

ENCODERS = [
    ('minilm', 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'),
    ('mmbert_base', 'jhu-clsp/mmbert-base'),
    ('e5_large', 'intfloat/multilingual-e5-large'),
    ('bge_m3', 'BAAI/bge-m3'),
]


def texts_for(variant, sep):
    def make(df):
        c = df['yt_comment'].astype(str).str.replace(r'\s+', ' ', regex=True).str.slice(0, 700)
        t = df['yt_title'].astype(str).str.replace(r'\s+', ' ', regex=True).str.slice(0, 140)
        d = df['yt_description'].astype(str).str.replace(r'\s+', ' ', regex=True).str.slice(0, 400)
        if variant == 'comment':
            return c.tolist()
        if variant == 'title+comment':
            return (t + f' {sep} ' + c).tolist()
        return (t + f' {sep} ' + d + f' {sep} ' + c).tolist()
    return make


@torch.no_grad()
def embed(model, tok, texts, bs=32):
    model.eval()
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], padding=True, truncation=True, max_length=MAXLEN,
                  return_tensors='pt')
        enc = {k: v.to(DEV) for k, v in enc.items()}
        h = model(**enc).last_hidden_state
        mask = enc['attention_mask'].unsqueeze(-1).to(h.dtype)
        v = (h * mask).sum(1) / mask.sum(1).clamp(min=1)
        v = F.normalize(v.float(), p=2, dim=-1)
        out.append(v.cpu().numpy().astype(np.float16))
    return np.concatenate(out, 0)


for tag, name in ENCODERS:
    need = [f'weights/probe_{tag}_{v}_{s}.npy'
            for v in ('comment', 'title+comment', 'title+desc+comment') for s in ('train', 'val')]
    if all(os.path.exists(p) for p in need):
        print('skip', tag, '(all embeddings present)', flush=True)
        continue
    try:
        cfg, cls = encoder_model_class(name)
        tok = AutoTokenizer.from_pretrained(name)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        try:
            model = cls.from_pretrained(name, config=cfg, dtype=torch.float16)
        except TypeError:
            model = cls.from_pretrained(name, config=cfg, torch_dtype=torch.float16)
        model.to(DEV)
    except Exception as e:  # noqa: BLE001
        print('LOAD FAILED', tag, name, '->', str(e)[:200], flush=True)
        continue
    sep = getattr(tok, 'sep_token', None) or ' | '
    print(f'== {tag} ({sum(p.numel() for p in model.parameters()) / 1e6:.0f}M, sep={sep!r})', flush=True)
    for variant in ('comment', 'title+comment', 'title+desc+comment'):
        make = texts_for(variant, sep)
        for split, df in (('train', df_train), ('val', df_val)):
            p = f'weights/probe_{tag}_{variant}_{split}.npy'
            if os.path.exists(p):
                print('  skip', p, flush=True)
                continue
            t0 = time.time()
            z = embed(model, tok, make(df))
            np.save(p, z)
            print(f'  {variant}/{split}: {z.shape} in {time.time() - t0:.0f}s', flush=True)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print('  freed', tag, flush=True)


## 3. Linear probe (fixed C, no val tuning)

Standardise on train, fit `LogisticRegression(C=0.1)`, evaluate the 738 val rows. C is fixed
before looking at val, so the comparison carries no selection bias.


In [ ]:
import json
import time

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

langs = np.load('weights/val_langs.npy').astype(str)
rows = []
for tag, _ in ENCODERS:
    for variant in ('comment', 'title+comment', 'title+desc+comment'):
        ptr = f'weights/probe_{tag}_{variant}_train.npy'
        pva = f'weights/probe_{tag}_{variant}_val.npy'
        if not (os.path.exists(ptr) and os.path.exists(pva)):
            continue
        X = np.load(ptr).astype(np.float32)
        Xv = np.load(pva).astype(np.float32)
        sc = StandardScaler().fit(X)
        t0 = time.time()
        clf = LogisticRegression(C=0.1, max_iter=2000, n_jobs=-1).fit(sc.transform(X), df_train['st_y'].astype(int))
        p = clf.predict_proba(sc.transform(Xv))[:, 1]
        pred = (p >= .5).astype(int)
        np.save(f'weights/probe_{tag}_{variant}_probs.npy', p.astype(np.float32))
        row = {
            'encoder': tag, 'variant': variant,
            'f1': float(f1_score(labs, pred, average='macro')),
            'acc': float((pred == labs).mean()),
            'pos_rate': float(pred.mean()),
        }
        for lg in ('EN', 'IT', 'NL'):
            sel = langs == lg
            row[f'f1_{lg}'] = float(f1_score(labs[sel], pred[sel], average='macro'))
        rows.append(row)
        print(f"{tag:12} {variant:22} F1 {row['f1']:.4f} acc {row['acc']:.4f} "
              f"pos {row['pos_rate']:.3f} ({time.time() - t0:.0f}s)", flush=True)

res = pd.DataFrame(rows).sort_values('f1', ascending=False)
print()
print(res.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
res.to_csv('weights/probe_encoders.csv', index=False)
with open('weights/probe_encoders.json', 'w') as fh:
    json.dump(rows, fh, indent=1)

ctrl = res[(res['encoder'] == 'minilm') & (res['variant'] == 'title+desc+comment')]
print()
if len(ctrl):
    base = float(ctrl['f1'].iloc[0])
    print(f'control (minilm, title+desc+comment): {base:.4f}')
    for _, r in res[res['variant'] == 'title+desc+comment'].iterrows():
        print(f"  {r['encoder']:12} {r['f1'] - base:+.4f} vs control")

import shutil
shutil.make_archive('/kaggle/working/probe_results', 'zip', 'weights')
print('zip MB:', os.path.getsize('/kaggle/working/probe_results.zip') / 1e6)
